# The Heritage Short — Wan 2.2 TI2V-5B moving clips (Kaggle)

**Kaggle settings:** Accelerator = **GPU T4 x2** (or T4), Internet = On.

Before running: on your Windows machine run `py run.py clips-export`, then upload `outputs/wan_job.zip` as a Kaggle dataset and set `KAGGLE_DATASET_NAME` below to that dataset slug. No API key is needed; the model weights are public and Apache 2.0.

**Do not run the final cell until you have viewed the smoke clip.** This stage is behind an explicit director approval gate. Output contract: `shot_01.mp4` … `shot_10.mp4` plus `wan_run_metadata.json`, zipped for `py run.py shots --import-dir <folder>`. **Fresh-session fix (2026-11-25):** the old `-U` install pulled transformers 5.x, whose `image_transforms` lacks `divide_to_patches`, so the Wan pipeline import failed. The cell now pins the tested trio (transformers 4.57.6 / hub 0.36.2 / diffusers 0.35.2), auto-detects a single uploaded dataset if the slug is still the placeholder, and guards later cells against a failed earlier cell (FAILURES.md 025).

In [ ]:
# 1. Locate the uploaded job and fail loudly if the contract is wrong.
import json, os, time
from pathlib import Path

# Safe defaults BEFORE any dataset/GPU call; later cells fail clearly if this aborts.
job = None
gpu = None
JOB_READY = False
MODEL_READY = False
records = []
load_sec = None

KAGGLE_DATASET_NAME = "REPLACE_WITH_YOUR_DATASET_SLUG"

if KAGGLE_DATASET_NAME.startswith('REPLACE_WITH'):
    found = [p for p in Path('/kaggle/input').glob('**/wan_job.json')]
    if len(found) == 1:
        KAGGLE_DATASET_NAME = found[0].relative_to('/kaggle/input').parts[0]
        print('Auto-detected Kaggle dataset slug:', KAGGLE_DATASET_NAME)
    elif not found:
        base = Path('/kaggle/input')
        visible = [p.name for p in base.iterdir()] if base.exists() else []
        print('No wan_job.json found. /kaggle/input contains:', visible)
JOB_DIR = next(Path('/kaggle/input').glob(f'{KAGGLE_DATASET_NAME}/**/wan_job.json'), None)
assert JOB_DIR is not None, (
    'wan_job.json not found. On Windows run `py run.py clips-export`, '
    'then upload outputs/wan_job.zip as a Kaggle dataset and set '
    'KAGGLE_DATASET_NAME to the slug shown in the Kaggle sidebar.'
)
JOB_DIR = JOB_DIR.parent
job = json.loads((JOB_DIR / 'wan_job.json').read_text(encoding='utf-8'))
assert len(job['shots']) == 10, 'Expected 10 shots in wan_job.json'
for s in job['shots']:
    assert (JOB_DIR / s['keyframe']).exists(), f"Missing keyframe {s['keyframe']}"

import torch
assert torch.cuda.is_available(), 'Set Kaggle Accelerator = GPU T4'
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, round(gpu.total_memory / 1024**3, 1), 'GB')
print('Model:', job['model'], '| shots:', len(job['shots']), '| fps:', job['fps'])
JOB_READY = True


In [ ]:
# 2. Install a diffusers build with WanImageToVideoPipeline, then ASSERT the import.
# We do not silently succeed: the ACE-Step notebook had a false success message once.
# Pinned to the tested trio: `-U` pulled transformers 5.x, whose image_transforms
# lacks divide_to_patches, so WanImageToVideoPipeline failed to import (FAILURES.md 025).
!pip install -q "transformers==4.57.6" "huggingface_hub==0.36.2" "diffusers==0.35.2" accelerate safetensors imageio imageio-ffmpeg ftfy
from importlib.metadata import version
for pkg, expected in (('transformers', '4.57.6'), ('huggingface_hub', '0.36.2'), ('diffusers', '0.35.2')):
    got = version(pkg)
    print(pkg, got)
    assert got == expected, f'{pkg} {got} != {expected}; see FAILURES.md 025'


from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import load_image
print('WanImageToVideoPipeline import: OK')

In [ ]:
# 3. Load once. Sequential CPU offload + VAE tiling keeps 480x480 within 15 GB T4.
if not JOB_READY:
    raise RuntimeError('Cell 1 did not locate a valid wan_job.json and GPU. Fix that cell first; no model was loaded.')
MODEL_ID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
t_load = time.perf_counter()
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.bfloat16,
)
assert pipe.transformer is not None and pipe.vae is not None, 'Model parts failed to load'
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
load_sec = round(time.perf_counter() - t_load, 1)
print(f'Models verified and loaded in {load_sec} s (offload enabled).')
MODEL_READY = True

In [ ]:
# 4. Helper: motion-only prompt (one gentle movement), fixed seed, 480x480 draft.
OUT_DIR = Path('/kaggle/working/wan_clips')
OUT_DIR.mkdir(parents=True, exist_ok=True)
STEPS = 20
GUIDANCE = 5.0


def generate_shot(shot):
    sid = shot['id']
    image = load_image(str(JOB_DIR / shot['keyframe'])).convert('RGB')
    width, height = int(job['width']), int(job['height'])
    assert width % 32 == 0 and height % 32 == 0, 'Wan needs dimensions divisible by 32'
    image = image.resize((width, height))
    frames_raw = round(float(shot['duration_sec']) * int(job['fps']))
    num_frames = ((frames_raw + 2) // 4) * 4 + 1  # Wan requires 4n+1 frames
    t0 = time.perf_counter()
    with torch.inference_mode():
        result = pipe(
            prompt=shot['video_motion_prompt'],
            negative_prompt=job['negative_prompt'],
            image=image,
            width=width,
            height=height,
            num_frames=num_frames,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            generator=torch.Generator(device='cuda').manual_seed(int(shot['video_seed'])),
        )
    frames = result.frames[0]
    assert len(frames) > 1, f'{sid}: no frames generated'
    out_path = OUT_DIR / f'{sid}.mp4'
    from diffusers.utils import export_to_video
    export_to_video(frames, str(out_path), fps=job['fps'])
    latency_ms = int((time.perf_counter() - t0) * 1000)
    return {
        'shot_id': sid, 'file_name': out_path.name, 'seed': int(shot['video_seed']),
        'num_frames': num_frames, 'fps': job['fps'], 'latency_ms': latency_ms,
        'model': MODEL_ID, 'status': 'ok',
    }

records = []
print('Helper ready. Generate the smoke clip in the next cell.')

In [ ]:
# 5. SMOKE TEST: one shot only. STOP afterwards and inspect the clip before continuing.
if not MODEL_READY:
    raise RuntimeError('The Wan pipeline was not verified in cells 2-3. Fix the import and load, then rerun; no clip was generated.')
smoke = generate_shot(job['shots'][0])
records = [smoke]
print('Smoke clip:', smoke)
import IPython.display as ipd
ipd.display(ipd.Video(str(OUT_DIR / smoke['file_name']), embed=True))
print('STOP: check face, hands and motion. Only run cell 6 if approved.')

In [ ]:
# 6. Only after director approval: the remaining nine shots, saving as we go.
import shutil
if not MODEL_READY:
    raise RuntimeError('Model was not verified; refusing to generate nine shots.')
assert len(records) == 1 and records[0]['shot_id'] == job['shots'][0]['id'], (
    'Run the smoke cell first (and only once).'
)
for shot in job['shots'][1:]:
    rec = generate_shot(shot)
    records.append(rec)
    print(rec['shot_id'], f"{rec['latency_ms']} ms", flush=True)

assert len(records) == 10
meta = {
    'model': MODEL_ID, 'load_sec': load_sec, 'steps': STEPS, 'guidance': GUIDANCE,
    'gpu': gpu.name, 'candidates': records,
}
meta_path = OUT_DIR / 'wan_run_metadata.json'
meta_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/kaggle/working/heritage_wan_clips', 'zip', OUT_DIR)
print('Download this zip and extract into a folder, then run: py run.py shots --import-dir <folder>')
print(zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Automatic download unavailable; use the Kaggle Output panel.')